# Tennis with Thompson

# USA Men's Tennis: 2026 Match-by-Match Analysis

This notebook builds analysis of **American men in ATP Masters 1000 and Grand Slam main draws during 2026**. It downloads TennisMyLife's completed-season and ongoing-tournament CSV files:

**Data source:** [TennisMyLife Tennis Match Database](https://stats.tennismylife.org/tennis-match-database).

## 1. Setup and configuration

NOTE: Change `SELECTED_PLAYER` later (in section 8) to drill into a different American player.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import io
import re
import sys
import shlex
import platform
import subprocess
import warnings

import numpy as np
import pandas as pd
import altair as alt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
alt.data_transformers.disable_max_rows()

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

URLS = {
    "completed": "https://stats.tennismylife.org/data/2026.csv",
    "ongoing": "https://stats.tennismylife.org/data/ongoing_tourneys.csv",
}

RELEVANT_LEVELS = {"G", "M"}  # Grand Slam and Masters 1000 in the actual 2026 files
EXPECTED_EVENTS = {
    "Australian Open", "Indian Wells Masters", "Miami Masters", "Monte Carlo",
    "Madrid Masters", "Rome Masters", "Roland Garros", "Wimbledon",
    "Canada Masters", "Cincinnati Masters", "US Open", "Shanghai Masters",
    "Paris Masters",
}

REFRESH_DATA = True
print(f"Congrats dude, the notebook ran")

## 2. Download completed and ongoing data

In [ ]:
def download_csv(label, url, refresh=True):
    cache_path = RAW_DIR / f"{label}_2026.csv"
    if refresh or not cache_path.exists():
        try:
            request = Request(url, headers={"User-Agent": "Mozilla/5.0 tennis-analysis-notebook"})
            with urlopen(request, timeout=45) as response:
                content = response.read()
            frame = pd.read_csv(io.BytesIO(content))
            frame.to_csv(cache_path, index=False)
            print(f"Downloaded {label}: {len(frame):,} rows")
            return frame
        except Exception as exc:
            if not cache_path.exists():
                raise RuntimeError(f"Could not download {label}, and no cache exists") from exc
            print(f"Download failed for {label}; using cached file ({exc})")
    frame = pd.read_csv(cache_path)
    print(f"Loaded cached {label}: {len(frame):,} rows")
    return frame


completed_raw = download_csv("completed", URLS["completed"], REFRESH_DATA)
ongoing_raw = download_csv("ongoing", URLS["ongoing"], REFRESH_DATA)
display(completed_raw.head(3))

## 3. Clean, validate, and deduplicate

Live records can overlap completed records. The pipeline prefers the ongoing version when keys overlap, then keeps the most complete record. The primary key is tournament ID plus match number; a fallback composite key handles missing match numbers.

In [ ]:
REQUIRED_COLUMNS = {
    "tourney_id", "tourney_name", "surface", "tourney_level", "tourney_date",
    "match_num", "winner_id", "winner_name", "winner_ioc", "winner_rank",
    "loser_id", "loser_name", "loser_ioc", "loser_rank", "score", "round",
    "minutes", "w_ace", "w_df", "w_svpt", "w_1stIn", "w_1stWon", "w_2ndWon",
    "w_bpSaved", "w_bpFaced", "l_ace", "l_df", "l_svpt", "l_1stIn",
    "l_1stWon", "l_2ndWon", "l_bpSaved", "l_bpFaced"
}

def validate_schema(frame, label):
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")

validate_schema(completed_raw, "completed")
validate_schema(ongoing_raw, "ongoing")

completed = completed_raw.assign(data_source="completed", source_priority=1)
ongoing = ongoing_raw.assign(data_source="ongoing", source_priority=2)
matches = pd.concat([completed, ongoing], ignore_index=True)

matches["match_date"] = pd.to_datetime(
    matches["tourney_date"].astype("Int64").astype(str), format="%Y%m%d", errors="coerce"
)
matches["year"] = matches["match_date"].dt.year
matches["row_completeness"] = matches.notna().sum(axis=1)
matches["fallback_key"] = (
    matches["tourney_id"].astype(str) + "|" + matches["round"].astype(str) + "|" +
    matches["winner_id"].astype(str) + "|" + matches["loser_id"].astype(str)
)
matches["match_key"] = np.where(
    matches["match_num"].notna(),
    matches["tourney_id"].astype(str) + "|" + matches["match_num"].astype(str),
    matches["fallback_key"]
)

rows_before = len(matches)
matches = (matches.sort_values(["source_priority", "row_completeness"])
           .drop_duplicates("match_key", keep="last")
           .reset_index(drop=True))
duplicates_removed = rows_before - len(matches)

# Restrict to the 2026 men's main-draw events requested.
big_events = matches[
    matches["year"].eq(2026)
    & matches["tourney_level"].isin(RELEVANT_LEVELS)
    & matches["tourney_name"].isin(EXPECTED_EVENTS)
].copy()

validation = pd.DataFrame({
    "check": ["Combined raw rows", "Duplicate rows removed", "Rows after deduplication",
              "Big-event matches", "Missing match dates", "Unexpected selected events"],
    "value": [rows_before, duplicates_removed, len(matches), len(big_events),
              int(big_events["match_date"].isna().sum()),
              int((~big_events["tourney_name"].isin(EXPECTED_EVENTS)).sum())]
})
display(validation)
display(big_events[["tourney_id", "tourney_name", "tourney_level", "surface"]]
        .drop_duplicates().sort_values("tourney_id").reset_index(drop=True))

## 4. Create the player-centric U.S. dataset

One row represents one American player's perspective on one match. An all-American match correctly produces two analytical rows. Break-point conversions are inferred from the opponent's break points faced minus break points saved.

In [ ]:
def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return numerator.div(denominator.where(denominator.ne(0)))


def player_perspective(frame, side):
    is_winner = side == "winner"
    p = "winner" if is_winner else "loser"
    o = "loser" if is_winner else "winner"
    ps = "w" if is_winner else "l"
    os = "l" if is_winner else "w"
    eligible = frame[f"{p}_ioc"].eq("USA")
    d = frame.loc[eligible].copy()

    out = pd.DataFrame({
        "match_key": d["match_key"], "match_date": d["match_date"], "year": d["year"],
        "tourney_id": d["tourney_id"], "tournament": d["tourney_name"],
        "event_type": np.where(d["tourney_level"].eq("G"), "Grand Slam", "Masters 1000"),
        "surface": d["surface"], "indoor": d["indoor"], "round": d["round"],
        "best_of": d["best_of"], "score": d["score"], "minutes": d["minutes"],
        "data_source": d["data_source"], "player_id": d[f"{p}_id"],
        "player": d[f"{p}_name"], "player_rank": d[f"{p}_rank"],
        "player_seed": d[f"{p}_seed"], "player_entry": d[f"{p}_entry"],
        "opponent_id": d[f"{o}_id"], "opponent": d[f"{o}_name"],
        "opponent_country": d[f"{o}_ioc"], "opponent_rank": d[f"{o}_rank"],
        "opponent_seed": d[f"{o}_seed"], "result": "W" if is_winner else "L",
        "win": int(is_winner), "aces": d[f"{ps}_ace"], "double_faults": d[f"{ps}_df"],
        "serve_points": d[f"{ps}_svpt"], "first_serves_in": d[f"{ps}_1stIn"],
        "first_serve_points_won": d[f"{ps}_1stWon"],
        "second_serve_points_won": d[f"{ps}_2ndWon"],
        "service_games": d[f"{ps}_SvGms"], "bp_saved": d[f"{ps}_bpSaved"],
        "bp_faced": d[f"{ps}_bpFaced"], "opp_bp_saved": d[f"{os}_bpSaved"],
        "opp_bp_faced": d[f"{os}_bpFaced"],
    })
    return out


usa = pd.concat([
    player_perspective(big_events, "winner"),
    player_perspective(big_events, "loser")
], ignore_index=True)

numeric_cols = [
    "player_rank", "opponent_rank", "minutes", "aces", "double_faults", "serve_points",
    "first_serves_in", "first_serve_points_won", "second_serve_points_won",
    "service_games", "bp_saved", "bp_faced", "opp_bp_saved", "opp_bp_faced"
]
usa[numeric_cols] = usa[numeric_cols].apply(pd.to_numeric, errors="coerce")

usa["ranking_difference"] = usa["opponent_rank"] - usa["player_rank"]
usa["upset_win"] = ((usa["win"] == 1) & (usa["player_rank"] > usa["opponent_rank"])).astype(int)
usa["upset_loss"] = ((usa["win"] == 0) & (usa["player_rank"] < usa["opponent_rank"])).astype(int)
usa["first_serve_in_pct"] = safe_divide(usa["first_serves_in"], usa["serve_points"])
usa["first_serve_win_pct"] = safe_divide(usa["first_serve_points_won"], usa["first_serves_in"])
second_serve_points = usa["serve_points"] - usa["first_serves_in"]
usa["second_serve_win_pct"] = safe_divide(usa["second_serve_points_won"], second_serve_points)
usa["bp_save_pct"] = safe_divide(usa["bp_saved"], usa["bp_faced"])
usa["bp_opportunities"] = usa["opp_bp_faced"]
usa["bp_converted"] = usa["opp_bp_faced"] - usa["opp_bp_saved"]
usa["bp_conversion_pct"] = safe_divide(usa["bp_converted"], usa["bp_opportunities"])
usa["ace_rate"] = safe_divide(usa["aces"], usa["serve_points"])
usa["double_fault_rate"] = safe_divide(usa["double_faults"], usa["serve_points"])

usa = usa.sort_values(["match_date", "tournament", "match_key", "player"]).reset_index(drop=True)
output_path = PROCESSED_DIR / "usa_men_big_events_2026.csv"
usa.to_csv(output_path, index=False)

print(f"Analytical rows: {len(usa):,}")
print(f"Unique matches involving a U.S. player: {usa['match_key'].nunique():,}")
print(f"American players: {usa['player'].nunique():,}")
print(f"Saved: {output_path}")
display(usa.head())

## 5. CLUTH TIME: deciding sets and tiebreaks

btw...it is called a tiebreak, not a tiebreaker (pet peeve)

In [ ]:
SET_PATTERN = re.compile(r"(?<!\d)(\d+)-(\d+)(?:\((\d+)\))?")

def parse_winner_score(score):
    text = "" if pd.isna(score) else str(score).strip().upper()
    sets = [(int(a), int(b), tb) for a, b, tb in SET_PATTERN.findall(text)]
    winner_sets = sum(a > b for a, b, _ in sets)
    loser_sets = sum(b > a for a, b, _ in sets)
    tiebreak_sets = [(a, b) for a, b, tb in sets if tb != ""]
    winner_tiebreaks = sum(a > b for a, b in tiebreak_sets)
    loser_tiebreaks = sum(b > a for a, b in tiebreak_sets)
    return pd.Series({
        "completed_sets": winner_sets + loser_sets,
        "winner_sets_won": winner_sets,
        "loser_sets_won": loser_sets,
        "tiebreaks_played": len(tiebreak_sets),
        "winner_tiebreaks_won": winner_tiebreaks,
        "loser_tiebreaks_won": loser_tiebreaks,
        "winner_won_first_set": bool(sets and sets[0][0] > sets[0][1]),
        "retirement_or_abandoned": any(flag in text for flag in ["RET", "ABD"]),
        "walkover_or_default": any(flag in text for flag in ["W/O", "WO", "DEF"]),
        "score_parseable": len(sets) > 0,
    })

score_features = big_events[["match_key", "best_of", "score"]].copy()
score_features = pd.concat(
    [score_features, score_features["score"].apply(parse_winner_score)], axis=1
)
score_features["deciding_set_played"] = (
    score_features["score_parseable"]
    & score_features["completed_sets"].eq(pd.to_numeric(score_features["best_of"], errors="coerce"))
)
score_features["five_set_match"] = (
    pd.to_numeric(score_features["best_of"], errors="coerce").eq(5)
    & score_features["completed_sets"].eq(5)
)

usa = usa.merge(
    score_features.drop(columns=["best_of", "score"]),
    on="match_key", how="left", validate="many_to_one"
)
usa["player_sets_won"] = np.where(usa["win"].eq(1), usa["winner_sets_won"], usa["loser_sets_won"])
usa["opponent_sets_won"] = np.where(usa["win"].eq(1), usa["loser_sets_won"], usa["winner_sets_won"])
usa["player_tiebreaks_won"] = np.where(
    usa["win"].eq(1), usa["winner_tiebreaks_won"], usa["loser_tiebreaks_won"]
)
usa["tiebreak_win_pct"] = safe_divide(usa["player_tiebreaks_won"], usa["tiebreaks_played"])
usa["straight_set_win"] = (
    usa["win"].eq(1) & usa["score_parseable"] & usa["opponent_sets_won"].eq(0)
).astype(int)
usa["deciding_set_win"] = (usa["deciding_set_played"] & usa["win"].eq(1)).astype(int)
usa["five_set_win"] = (usa["five_set_match"] & usa["win"].eq(1)).astype(int)
usa["comeback_win"] = (
    usa["win"].eq(1) & usa["score_parseable"] & ~usa["winner_won_first_set"]
).astype(int)

score_quality = pd.DataFrame({
    "metric": ["Rows with parseable scores", "Deciding-set rows", "Five-set rows",
               "Rows with a tiebreak", "Retirement/abandoned rows", "Walkover/default rows"],
    "value": [usa["score_parseable"].sum(), usa["deciding_set_played"].sum(),
              usa["five_set_match"].sum(), usa["tiebreaks_played"].gt(0).sum(),
              usa["retirement_or_abandoned"].sum(), usa["walkover_or_default"].sum()]
})
display(score_quality)

## 6. Ranking-based expectations

The player's estimated probability is `opponent_rank / (player_rank + opponent_rank)`. It uses only the two match-time rankings, not the match result. This is an interpretable benchmark. It is not a fully calibrated forecasting model.

- **Expected win:** probability at least 65%
- **Toss-up:** probability between 35% and 65%
- **Expected loss:** probability at most 35%
- **Performance versus expectation:** actual result (1/0) minus expected probability

In [ ]:
valid_ranks = usa["player_rank"].gt(0) & usa["opponent_rank"].gt(0)
usa["ranking_expected_win_prob"] = np.where(
    valid_ranks,
    usa["opponent_rank"] / (usa["player_rank"] + usa["opponent_rank"]),
    np.nan,
)
usa["expectation_group"] = pd.cut(
    usa["ranking_expected_win_prob"],
    bins=[-np.inf, 0.35, 0.65, np.inf],
    labels=["Expected loss", "Toss-up", "Expected win"],
    right=False,
)
usa["performance_vs_expectation"] = usa["win"] - usa["ranking_expected_win_prob"]
usa["expectation_outcome"] = np.select(
    [
        usa["win"].eq(1) & usa["expectation_group"].eq("Expected loss"),
        usa["win"].eq(0) & usa["expectation_group"].eq("Expected win"),
        usa["win"].eq(1),
    ],
    ["Upset win", "Unexpected loss", "Win"],
    default="Loss",
)

expectation_summary = (usa.groupby("player", as_index=False, observed=True)
    .agg(matches=("win", "size"), actual_win_pct=("win", "mean"),
         expected_win_pct=("ranking_expected_win_prob", "mean"),
         performance_vs_expectation=("performance_vs_expectation", "mean"),
         upset_wins=("expectation_outcome", lambda s: s.eq("Upset win").sum()),
         unexpected_losses=("expectation_outcome", lambda s: s.eq("Unexpected loss").sum()))
    .sort_values("performance_vs_expectation", ascending=False))

display(expectation_summary.round(3))

# Re-export after adding score and expectation features.
usa.to_csv(output_path, index=False)
print(f"Updated analytical dataset: {output_path}")

## 7. Player summary table

In [ ]:
summary = (usa.groupby("player", as_index=False)
           .agg(matches=("match_key", "count"), wins=("win", "sum"),
                win_pct=("win", "mean"), tournaments=("tournament", "nunique"),
                best_rank=("player_rank", "min"), upsets=("upset_win", "sum"),
                avg_aces=("aces", "mean"), first_serve_in_pct=("first_serve_in_pct", "mean"),
                first_serve_win_pct=("first_serve_win_pct", "mean"),
                second_serve_win_pct=("second_serve_win_pct", "mean"),
                bp_save_pct=("bp_save_pct", "mean"),
                bp_conversion_pct=("bp_conversion_pct", "mean"),
                deciding_sets=("deciding_set_played", "sum"),
                deciding_set_wins=("deciding_set_win", "sum"),
                tiebreaks=("tiebreaks_played", "sum"),
                tiebreaks_won=("player_tiebreaks_won", "sum"),
                comeback_wins=("comeback_win", "sum"),
                expected_win_pct=("ranking_expected_win_prob", "mean"),
                performance_vs_expectation=("performance_vs_expectation", "mean"))
           .assign(losses=lambda x: x["matches"] - x["wins"])
           .sort_values(["wins", "win_pct", "matches"], ascending=False))

pct_cols = ["win_pct", "first_serve_in_pct", "first_serve_win_pct",
            "second_serve_win_pct", "bp_save_pct", "bp_conversion_pct",
            "expected_win_pct", "performance_vs_expectation"]
summary_display = summary.copy()
summary_display[pct_cols] = (summary_display[pct_cols] * 100).round(1)
display(summary_display.rename(columns={c: f"{c} (%)" for c in pct_cols}))

## 8. Match-by-match drill-down

Choose any name from `sorted(usa.player.unique())`. The table includes the requested opponent, result, event context, rankings, serve statistics, break-point measures, duration, and score.

In [ ]:
print(sorted(usa["player"].unique()))
SELECTED_PLAYER = "Tommy Paul"

detail_columns = [
    "match_date", "player", "opponent", "opponent_country", "result", "tournament",
    "event_type", "surface", "round", "player_rank", "opponent_rank",
    "ranking_difference", "upset_win", "aces", "double_faults", "first_serve_in_pct",
    "first_serve_win_pct", "second_serve_win_pct", "bp_save_pct", "bp_converted",
    "bp_opportunities", "bp_conversion_pct", "player_sets_won", "opponent_sets_won",
    "tiebreaks_played", "player_tiebreaks_won", "deciding_set_played", "comeback_win",
    "ranking_expected_win_prob", "expectation_group", "performance_vs_expectation",
    "minutes", "score", "data_source"
]
player_matches = usa.loc[usa["player"].eq(SELECTED_PLAYER), detail_columns]
player_matches_display = player_matches.sort_values("match_date", ascending=False).copy()
detail_pct_cols = ["first_serve_in_pct", "first_serve_win_pct", "second_serve_win_pct",
                   "bp_save_pct", "bp_conversion_pct", "ranking_expected_win_prob",
                   "performance_vs_expectation"]
player_matches_display[detail_pct_cols] = (player_matches_display[detail_pct_cols] * 100).round(1)
display(player_matches_display.rename(columns={c: f"{c} (%)" for c in detail_pct_cols}))

## 9. Interactive Altair visualizations

All charts support interactive tooltips. The time-series chart also supports zooming and horizontal panning.

In [ ]:
# Win percentage by player (minimum 3 matches)
plot_summary = summary.query("matches >= 3").copy()

win_chart = (
    alt.Chart(plot_summary)
    .mark_bar()
    .encode(
        x=alt.X("win_pct:Q", title="Win percentage", axis=alt.Axis(format=".0%")),
        y=alt.Y("player:N", title=None, sort="-x"),
        color=alt.Color("matches:Q", title="Matches", scale=alt.Scale(scheme="viridis")),
        tooltip=[
            alt.Tooltip("player:N", title="Player"),
            alt.Tooltip("matches:Q", title="Matches"),
            alt.Tooltip("wins:Q", title="Wins"),
            alt.Tooltip("losses:Q", title="Losses"),
            alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
        ],
    )
    .properties(
        width=700,
        height=max(350, 24 * len(plot_summary)),
        title="2026 Win Percentage: U.S. Men in Grand Slams and Masters 1000",
    )
)
win_chart

In [ ]:
# Surface performance for players with at least 3 matches on a surface
surface = (usa.groupby(["player", "surface"], as_index=False)
           .agg(matches=("win", "size"), wins=("win", "sum"), win_pct=("win", "mean"))
           .query("matches >= 3"))
player_order = summary.sort_values("wins", ascending=False)["player"].tolist()

surface_base = alt.Chart(surface).encode(
    x=alt.X("surface:N", title=None),
    y=alt.Y("player:N", title=None, sort=player_order),
    tooltip=[
        alt.Tooltip("player:N", title="Player"),
        alt.Tooltip("surface:N", title="Surface"),
        alt.Tooltip("matches:Q", title="Matches"),
        alt.Tooltip("wins:Q", title="Wins"),
        alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
    ],
)
surface_chart = (
    surface_base.mark_rect()
    .encode(color=alt.Color("win_pct:Q", title="Win %", scale=alt.Scale(scheme="redyellowgreen", domain=[0, 1])))
    + surface_base.mark_text(fontSize=11).encode(
        text=alt.Text("win_pct:Q", format=".0%"),
        color=alt.condition("datum.win_pct > 0.75 || datum.win_pct < 0.25", alt.value("white"), alt.value("black")),
    )
).properties(
    width=400,
    height=max(350, 24 * surface["player"].nunique()),
    title="Win Percentage by Surface (minimum 3 matches)",
)
surface_chart

In [ ]:
# Serve effectiveness: first-serve points won vs second-serve points won
serve = (usa.groupby("player", as_index=False)
         .agg(matches=("match_key", "size"), first_serve_win_pct=("first_serve_win_pct", "mean"),
              second_serve_win_pct=("second_serve_win_pct", "mean"), win_pct=("win", "mean"))
         .query("matches >= 5").dropna())
serve["short_name"] = serve["player"].str.split().str[-1]

serve_points = alt.Chart(serve).mark_circle(opacity=.85, stroke="white", strokeWidth=1).encode(
    x=alt.X("first_serve_win_pct:Q", title="First-serve points won", axis=alt.Axis(format=".0%"), scale=alt.Scale(zero=False)),
    y=alt.Y("second_serve_win_pct:Q", title="Second-serve points won", axis=alt.Axis(format=".0%"), scale=alt.Scale(zero=False)),
    size=alt.Size("matches:Q", title="Matches", scale=alt.Scale(range=[80, 800])),
    color=alt.Color("win_pct:Q", title="Win %", scale=alt.Scale(scheme="viridis")),
    tooltip=[
        alt.Tooltip("player:N", title="Player"),
        alt.Tooltip("matches:Q", title="Matches"),
        alt.Tooltip("first_serve_win_pct:Q", title="1st serve won", format=".1%"),
        alt.Tooltip("second_serve_win_pct:Q", title="2nd serve won", format=".1%"),
        alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
    ],
)
serve_labels = alt.Chart(serve).mark_text(dx=10, dy=-8, fontSize=10).encode(
    x="first_serve_win_pct:Q", y="second_serve_win_pct:Q", text="short_name:N"
)
serve_chart = (serve_points + serve_labels).properties(
    width=700, height=450, title="Serve Effectiveness (minimum 5 matches)"
)
serve_chart

In [ ]:
# Ranking upsets
upsets = (usa.groupby("player", as_index=False)["upset_win"].sum()
          .query("upset_win > 0"))

upset_chart = (
    alt.Chart(upsets)
    .mark_bar(color="#2a6fbb")
    .encode(
        x=alt.X("upset_win:Q", title="Upset wins", axis=alt.Axis(tickMinStep=1)),
        y=alt.Y("player:N", title=None, sort="-x"),
        tooltip=[alt.Tooltip("player:N", title="Player"), alt.Tooltip("upset_win:Q", title="Upset wins")],
    )
    .properties(width=650, height=max(300, 24 * len(upsets)), title="Wins over Higher-Ranked Opponents")
)
upset_chart

In [ ]:
# Tournament results: wins per player and event
tourney = (usa.groupby(["player", "tournament"], as_index=False)
           .agg(matches=("win", "size"), wins=("win", "sum")))
player_order = summary.sort_values("wins", ascending=False)["player"].tolist()

tourney_base = alt.Chart(tourney).encode(
    x=alt.X("tournament:N", title=None, sort=None, axis=alt.Axis(labelAngle=-40)),
    y=alt.Y("player:N", title=None, sort=player_order),
    tooltip=[
        alt.Tooltip("player:N", title="Player"),
        alt.Tooltip("tournament:N", title="Tournament"),
        alt.Tooltip("matches:Q", title="Matches"),
        alt.Tooltip("wins:Q", title="Wins"),
    ],
)
tourney_chart = (
    tourney_base.mark_rect().encode(
        color=alt.Color("wins:Q", title="Match wins", scale=alt.Scale(scheme="blues"))
    )
    + tourney_base.mark_text(fontSize=10).encode(
        text=alt.Text("wins:Q", format=".0f"),
        color=alt.condition("datum.wins >= 4", alt.value("white"), alt.value("black")),
    )
).properties(
    width=700,
    height=max(400, 24 * tourney["player"].nunique()),
    title="Tournament Results: Match Wins",
)
tourney_chart

In [ ]:
# Performance over time for the leading players by total wins
leaders = summary.head(8)["player"].tolist()
trend = usa[usa["player"].isin(leaders)].sort_values(["player", "match_date", "match_key"]).copy()
trend["match_number"] = trend.groupby("player").cumcount() + 1
trend["cumulative_win_pct"] = trend.groupby("player")["win"].cumsum() / trend["match_number"]

trend_chart = (
    alt.Chart(trend)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X("match_date:T", title="Match date"),
        y=alt.Y("cumulative_win_pct:Q", title="Cumulative win percentage", axis=alt.Axis(format=".0%"), scale=alt.Scale(domain=[0, 1])),
        color=alt.Color("player:N", title="Player"),
        tooltip=[
            alt.Tooltip("match_date:T", title="Date"),
            alt.Tooltip("player:N", title="Player"),
            alt.Tooltip("match_number:Q", title="Match number"),
            alt.Tooltip("cumulative_win_pct:Q", title="Cumulative win %", format=".1%"),
        ],
    )
    .properties(width=750, height=450, title="Cumulative Win Percentage Over 2026")
    .interactive(bind_y=False)
)
trend_chart

In [ ]:
# Deciding-set and tiebreak performance
clutch = (usa.groupby("player", as_index=False)
          .agg(matches=("win", "size"), deciding_sets=("deciding_set_played", "sum"),
               deciding_set_wins=("deciding_set_win", "sum"),
               tiebreaks=("tiebreaks_played", "sum"),
               tiebreaks_won=("player_tiebreaks_won", "sum")))
clutch["deciding_set_win_pct"] = safe_divide(clutch["deciding_set_wins"], clutch["deciding_sets"])
clutch["tiebreak_win_pct"] = safe_divide(clutch["tiebreaks_won"], clutch["tiebreaks"])
clutch_long = clutch.melt(
    id_vars=["player", "matches", "deciding_sets", "tiebreaks"],
    value_vars=["deciding_set_win_pct", "tiebreak_win_pct"],
    var_name="metric", value_name="rate",
).dropna()
clutch_long["metric"] = clutch_long["metric"].map({
    "deciding_set_win_pct": "Deciding sets", "tiebreak_win_pct": "Tiebreaks"
})

clutch_chart = (
    alt.Chart(clutch_long)
    .mark_bar()
    .encode(
        x=alt.X("rate:Q", title="Win percentage", axis=alt.Axis(format=".0%")),
        y=alt.Y("player:N", title=None, sort="-x"),
        color=alt.Color("metric:N", title="Metric"),
        yOffset="metric:N",
        tooltip=[
            alt.Tooltip("player:N", title="Player"), alt.Tooltip("metric:N", title="Metric"),
            alt.Tooltip("rate:Q", title="Win %", format=".1%"),
            alt.Tooltip("deciding_sets:Q", title="Deciding sets"),
            alt.Tooltip("tiebreaks:Q", title="Tiebreaks"),
        ],
    )
    .properties(width=650, height=max(350, 28 * clutch_long["player"].nunique()),
                title="Deciding-Set and Tiebreak Performance")
)
clutch_chart

In [ ]:
# Actual versus ranking-expected win percentage (minimum 5 matches)
expectation_plot = expectation_summary.query("matches >= 5").copy()
expectation_long = expectation_plot.melt(
    id_vars=["player", "matches", "performance_vs_expectation"],
    value_vars=["actual_win_pct", "expected_win_pct"],
    var_name="metric", value_name="win_pct",
)
expectation_long["metric"] = expectation_long["metric"].map({
    "actual_win_pct": "Actual", "expected_win_pct": "Ranking expected"
})
expectation_order = expectation_plot.sort_values("performance_vs_expectation", ascending=False)["player"].tolist()

expectation_chart = (
    alt.Chart(expectation_long)
    .mark_bar()
    .encode(
        x=alt.X("player:N", title=None, sort=expectation_order, axis=alt.Axis(labelAngle=-45)),
        xOffset="metric:N",
        y=alt.Y("win_pct:Q", title="Win percentage", axis=alt.Axis(format=".0%")),
        color=alt.Color("metric:N", title=None),
        tooltip=[
            alt.Tooltip("player:N", title="Player"), alt.Tooltip("metric:N", title="Measure"),
            alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
            alt.Tooltip("performance_vs_expectation:Q", title="Over/under expectation", format="+.1%"),
            alt.Tooltip("matches:Q", title="Matches"),
        ],
    )
    .properties(width=750, height=420,
                title="Actual vs Ranking-Expected Win Percentage (minimum 5 matches)")
)
expectation_chart

## 10. Additional validation and data-quality notes

- Missing serve statistics remain missing; they are not replaced with zero.
- A break-point percentage is missing when no relevant opportunities occurred.
- Rankings are match-time ATP rankings.
- `ranking_difference = opponent_rank - player_rank`; negative values mean the American was ranked lower.
- Ranking-based probabilities are a transparent ordinal-rank benchmark, not calibrated betting probabilities.
- Score-derived statistics exclude unparseable sets; retirements and walkovers remain explicitly flagged.
- Live data may be revised by the provider. Rerunning the notebook refreshes and deduplicates it.
- Events not yet played in 2026 do not appear until the source publishes matches.

In [ ]:
quality_report = pd.DataFrame({
    "metric": [
        "Analytical rows", "Unique qualifying matches", "U.S. players", "All-American matches",
        "Missing player rankings", "Missing opponent rankings", "Missing serve points",
        "Missing match durations", "Duplicate player-match rows"
    ],
    "value": [
        len(usa), usa["match_key"].nunique(), usa["player"].nunique(),
        big_events.eval("winner_ioc == 'USA' and loser_ioc == 'USA'").sum(),
        usa["player_rank"].isna().sum(), usa["opponent_rank"].isna().sum(),
        usa["serve_points"].isna().sum(), usa["minutes"].isna().sum(),
        usa.duplicated(["match_key", "player_id"]).sum()
    ]
})
display(quality_report)

assert usa.duplicated(["match_key", "player_id"]).sum() == 0
assert set(usa["result"].dropna().unique()) <= {"W", "L"}
assert usa["player"].notna().all()
assert usa["tournament"].isin(EXPECTED_EVENTS).all()
assert usa["year"].eq(2026).all()
print("All final assertions passed.")